# COPY INTO y Comparacion

# COPY INTO: ingesta incremental con SQL 

El mismo problema del notebook 61, cargar solo los archivos nuevos, sin duplicar, resuelto con un solo comando SQL.

`COPY INTO` es un comando **reintentable e idempotente** que carga archivos desde object
storage (aquí, nuestro Volume) hacia una tabla Delta, **rastreando qué archivos ya
cargó**. A diferencia de Auto Loader, exige que la tabla destino exista.

**Estatus:** Databricks lo considera *legacy* y recomienda Auto Loader para código nuevo,
pero sigue en el examen DCEA y es común en pipelines existentes. Carga ad-hoc o programadas con **miles** de archivos (Auto Loader escala a miles de millones).

In [ ]:
%run ./60_Setup_Landing_Zone

In [ ]:
# Limpieza local idempotente: este notebook resetea SU propio estado.
dbutils.fs.rm(LANDING_CI, recurse=True)
dbutils.fs.mkdirs(LANDING_CI)

spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{SCHEMA}.ratings_bronze_ci")
print("Estado del notebook 62 reiniciado.")

In [ ]:
# El "sistema externo" deposita el lote 1 en CSV
generar_lote_csv(LANDING_CI, lote=1, n=500)
display(dbutils.fs.ls(LANDING_CI))

In [ ]:
%sql
-- COPY INTO exige que la tabla destino exista.
-- La creamos VACÍA y SIN columnas (fíjate: no declaramos ningún esquema).
CREATE TABLE IF NOT EXISTS ratings_bronze_ci;

COPY INTO exige que la tabla destino exista. Acabamos de crearla vacía y **sin columnas**. ¿De dónde va a salir el esquema cuando ejecutemos COPY INTO?

In [ ]:
%sql
COPY INTO ratings_bronze_ci
FROM '/Volumes/big_data_ii_2025/spark_examples/landing/copyinto/'
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'true', 'mergeSchema' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true')

El esquema salió de los **archivos**: `inferSchema` lee los tipos del CSV y el
`mergeSchema` de `COPY_OPTIONS` deja que la tabla vacía adopte (y luego evolucione) ese
esquema. El resultado del comando reporta `num_affected_rows` / `num_inserted_rows`.

In [ ]:
%sql
SELECT COUNT(*) AS filas FROM ratings_bronze_ci

Ejecutamos el MISMO `COPY INTO` otra vez, sin archivos nuevos. ¿Cuál será `num_inserted_rows`?

In [ ]:
%sql
-- Exactamente el mismo comando de arriba
COPY INTO ratings_bronze_ci
FROM '/Volumes/big_data_ii_2025/spark_examples/landing/copyinto/'
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'true', 'mergeSchema' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true')

**0 filas insertadas.** COPY INTO rastrea los archivos ya cargados: `lote_01.csv` ya
está en su historial, así que lo ignora. Idempotencia por rastreo de archivos — re-ejecutar
es seguro.

In [ ]:
# Llega el lote 2
generar_lote_csv(LANDING_CI, lote=2, n=500)
display(dbutils.fs.ls(LANDING_CI))

Con el lote 2 en el landing, ¿cuántas filas tendrá la tabla después del próximo COPY INTO?

In [ ]:
%sql
COPY INTO ratings_bronze_ci
FROM '/Volumes/big_data_ii_2025/spark_examples/landing/copyinto/'
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'true', 'mergeSchema' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true')

In [ ]:
%sql
SELECT COUNT(*) AS filas FROM ratings_bronze_ci

El rastreo es **por archivo** (por nombre). Si el proveedor
*modifica* un archivo ya cargado y lo vuelve a subir **con el mismo nombre**, COPY INTO lo
ignora — no detecta el cambio. Tanto COPY INTO como Auto Loader asumen **archivos
inmutables**: una corrección se publica como un archivo nuevo con nombre nuevo.

**Modo `VALIDATE`:** permite previsualizar la carga sin ejecutarla (revisa esquema y
muestra filas de ejemplo). No lo ejecutamos aquí, pero la sintaxis es:

```sql
COPY INTO ratings_bronze_ci
FROM '/Volumes/big_data_ii_2025/spark_examples/landing/copyinto/'
FILEFORMAT = CSV
VALIDATE
FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'true')
```

## Comparación: ¿cuál mecanismo elijo?

| Criterio | Auto Loader | COPY INTO | Lakeflow Connect (managed) |
|---|---|---|---|
| Interfaz | Python (Structured Streaming) | SQL | UI / API, sin código |
| Escala de archivos | Miles de millones | Miles | N/A — fuentes SaaS y BD (APIs / logs) |
| Evolución de esquema | Modos + `_rescued_data` | `mergeSchema` | Automática |
| Estado ("memoria") | Checkpoint (RocksDB) | Historial interno de archivos | Gestionado por la plataforma |
| Estatus | **Recomendado** | Legacy (vigente en el examen) | Recomendado para fuentes empresariales |
| En Free Edition | Sí | Sí | Solo conceptual |